## Open A PDF

In [1]:
import fitz

In [2]:
pdf = fitz.open("documents/Day_1_v3.pdf")
print("Number of pages:",len(pdf))

Number of pages: 51


In [4]:
for page_number,page in enumerate(pdf):
    text = page.get_text()
    print(f"PAGE {page_number+1}")
    print(text)

PAGE 1
The New SDLC  
With Vibe Coding 
 
 
Authors: Addy Osmani, Shubham Saboo,  
and Sokratis Kartakis
From ad-hoc prompting to  
Agentic Engineering

PAGE 2
The new SDLC with vibe coding
May 2026
2
Acknowledgements
Content contributors
Elia Secchi
Julia Wiesinger
Anant Nawalgaria
Curators and editors
Anant Nawalgaria
Designer
Michael Lanning 

PAGE 3
Introductio
6
Why this paper, why no
9
Who this paper is fo
9
The shift from syntax to inten
10
AI Agents: A Quick Refreshe
10
What is vibe coding
11
The spectrum: vibe coding to agentic engineerin
12
Context engineering: the real skil
15
The new software development life cycl
19
The traditional SDLC under pressur
19
How AI transforms each phas
21
Requirements and planning	
21
Design and architecture	
21
Implementation	
22
Testing and quality assurance	
22
Table of contents

PAGE 4
Code review and deployment	
23
Maintenance and evolution	
24
The factory model: building the system that builds softwar
24
Harness Engineering: What surround

## Extract text block

In [5]:
page = pdf[0]


In [7]:
blocks = page.get_text('blocks')

In [8]:
for block in blocks:
    print(block)

(35.999969482421875, 129.22378540039062, 535.2599487304688, 423.4997863769531, 'The New SDLC  \nWith Vibe Coding \n \n \nAuthors: Addy Osmani, Shubham Saboo,  \nand Sokratis Kartakis\n', 0, 0)
(36.0, 260.8816223144531, 352.4663391113281, 320.85760498046875, 'From ad-hoc prompting to  \nAgentic Engineering\n', 1, 0)


In [10]:
for block in page.get_text('blocks'):
    x0,y0,x1,y1, text, *_ = block
    print("TEXT:",text)
    print("BBOX:",[x0,y0,x1,y1])

TEXT: The New SDLC  
With Vibe Coding 
 
 
Authors: Addy Osmani, Shubham Saboo,  
and Sokratis Kartakis

BBOX: [35.999969482421875, 129.22378540039062, 535.2599487304688, 423.4997863769531]
TEXT: From ad-hoc prompting to  
Agentic Engineering

BBOX: [36.0, 260.8816223144531, 352.4663391113281, 320.85760498046875]


In [11]:
images = page.get_images(full=True)
print(len(images))

1


In [12]:
images

[(578, 577, 1559, 1179, 8, 'ICCBased', '', 'Im0', 'JPXDecode', 0)]

## Render a Images

In [14]:
page = pdf[1]

In [16]:
pix = page.get_pixmap(matrix=fitz.Matrix(2,2))
pix.save("page_002.png")

## Using pdfplumber 

In [17]:
pip install pdfplumber

Note: you may need to restart the kernel to use updated packages.


In [18]:
import pdfplumber

In [24]:
li = []
with  pdfplumber.open("documents/Agent Tools & Interoperability_Day_2.pdf") as pdf:
    for page_number,page in enumerate(pdf.pages):
        tables = page.extract_tables()
        print(
            f"Page {page_number + 1}: "
            f"{len(tables)} tables"
        )
        li.append(tables)

Page 1: 0 tables
Page 2: 0 tables
Page 3: 0 tables
Page 4: 0 tables
Page 5: 0 tables
Page 6: 0 tables
Page 7: 0 tables
Page 8: 0 tables
Page 9: 0 tables
Page 10: 0 tables
Page 11: 0 tables
Page 12: 0 tables
Page 13: 0 tables
Page 14: 0 tables
Page 15: 0 tables
Page 16: 0 tables
Page 17: 0 tables
Page 18: 0 tables
Page 19: 0 tables
Page 20: 0 tables
Page 21: 0 tables
Page 22: 0 tables
Page 23: 0 tables
Page 24: 0 tables
Page 25: 0 tables
Page 26: 0 tables
Page 27: 0 tables
Page 28: 0 tables
Page 29: 0 tables
Page 30: 0 tables
Page 31: 0 tables
Page 32: 0 tables
Page 33: 0 tables
Page 34: 1 tables
Page 35: 0 tables
Page 36: 0 tables
Page 37: 1 tables
Page 38: 0 tables
Page 39: 0 tables
Page 40: 0 tables
Page 41: 0 tables
Page 42: 0 tables
Page 43: 0 tables
Page 44: 0 tables
Page 45: 0 tables
Page 46: 1 tables
Page 47: 0 tables
Page 48: 0 tables
Page 49: 0 tables


In [22]:
tables

[]

In [26]:
for item in li:
    for table in item:
        for row in table:
            print(row)

['Layout', 'Row, Column, List']
['Display', 'Text, Image, Icon, Divider']
['Containers', 'Card, Modal, Tabs']
['Media', 'Video, AudioPlayer']
['"What\'s the average?"', 'Data (text)']
['"Compare these regions"', 'UI generated by the LLM (§4.4)']
['"Show me my dashboard"', 'UI built by a tool (this section)']
['Brain that decides what to buy, handles the menu,\nand puts the food in the cart.', 'Is the wallet that securely handles how to pay\nfor it without you getting scammed.']
['Integrates with any business provider', 'Integrates with payment']


In [28]:
table_data = {
    'type':'table',
    'columns':table[0],
    'row':table[1:]
}

In [29]:
table_data

{'type': 'table',
 'columns': ['Brain that decides what to buy, handles the menu,\nand puts the food in the cart.',
  'Is the wallet that securely handles how to pay\nfor it without you getting scammed.'],
 'row': [['Integrates with any business provider', 'Integrates with payment']]}

## Build our first extractor

In [31]:
import fitz
import pdfplumber
import os 
import json

class PDFExtractor:
    def __init__(self,pdf_path:str,output_dir:str|None = "output"):
        self.pdf_path = pdf_path
        self.output_dir = output_dir
        self.pdf = fitz.open(pdf_path)
        os.makedirs(output_dir,exist_ok=True)
        os.makedirs(os.path.join(output_dir,"images"),exist_ok=True)

    def extract(self):
        document = {
            "document_id":os.path.splitext(
                os.path.basename(self.pdf_path)
            )[0],
            'filename':os.path.basename(self.pdf_path),
            'num_pages':len(self.pdf),
            'elements':[]
        }
        self.extract_text(document)
        self.extract_images(document)
        self.extract_tables(document)
        return document

    def extract_text(self,document):
        for page_number, page in enumerate(self.pdf):
            blocks = page.get_text('blocks')
            for index, block in enumerate(blocks):
                x0,y0,x1,y1,text,*_ = block
                text = text.strip()
                if not text:
                    continue
                element = {
                    "id":(
                        f"{document['document_id']}"
                        f"_p{page_number+1}"
                        f"_text{index+1}"
                    ),
                    'document_id':document['document_id'],
                    'page_number':page_number +1 ,
                    'element_type':'text',
                    'content':text,
                    'bbox':[x0,y0,x1,y1],
                    'path':None,
                    'metadata':{}
                }
                document['elements'].append(element)